[Reference](https://blogs.learningdevops.com/i-built-a-bpe-tokenizer-from-scratch-in-python-and-finally-understand-how-llms-work-4d1034d264ed$0)

In [1]:
# -----Functions--------------------------------------------------------------

def get_stats(ids):
    counts = {}
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i+1])
        counts[pair] = counts.get(pair, 0) + 1
    return counts
def merge(ids, pair, idx):
    new_ids_list = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            new_ids_list.append(idx)
            i += 2
        else:
            new_ids_list.append(ids[i])
            i += 1
    return new_ids_list
def get_merge_index(p):
    return merges.get(p, float("inf"))
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8", errors="replace")
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=get_merge_index)
        if pair not in merges:
            break
        idx = merges[pair]
        ids = merge(ids, pair, idx)
    return ids

# -- Step 1: Text to raw bytes -----------------------------------------------
print("=" * 60)
print("STEP 1: Text to raw bytes")
print("=" * 60)
sample = "hello world"
print(f"Text: {sample}")
print(f"Raw bytes: {list(sample.encode('utf-8'))}")

# -- Step 2: get_stats demo --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 2: Finding frequent pairs")
print("=" * 60)
demo_ids = [1, 2, 3, 1, 2]
print(f"Input ids: {demo_ids}")
print(f"Pair counts: {get_stats(demo_ids)}")

# -- Step 3: merge demo --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 3: Merging a pair")
print("=" * 60)
print(f"Before merge: {demo_ids}")
print(f"After merging (1,2) -> 99: {merge(demo_ids, (1, 2), 99)}")

# -- Step 4: Training loop --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 4: Training on real text")
print("=" * 60)
training_text = """Linux is a family of open-source Unix-like operating systems based on the Linux kernel,
first released by Linus Torvalds on September 17, 1991. Linux is typically packaged as a Linux
distribution, which includes the kernel and supporting system software and libraries,
many of which are provided by the GNU Project. Many Linux distributions use the word Linux
in their name, but the Free Software Foundation uses the name GNU/Linux to emphasize the
importance of GNU software, causing some controversy."""
ids = list(training_text.encode("utf-8"))
original_len = len(ids)
print(f"Training text length in raw bytes: {original_len}")
print(f"Running 20 merges...\n")
merges = {}
for i in range(20):
    stats = get_stats(ids)
    top_pair = max(stats, key=stats.get)
    ids = merge(ids, top_pair, 256 + i)
    merges[top_pair] = 256 + i
    print(f"Merge {i+1:2d}: {top_pair} -> {256+i}  (length now {len(ids)})")
compressed_training_ids = ids
print(f"\nLearned merges: {merges}")
print(f"\nOriginal length: {original_len}")
print(f"Compressed length: {len(compressed_training_ids)}")
print(f"First 20 compressed token ids: {compressed_training_ids[:20]}")

# -- Step 5: Build vocab --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 5: Building vocabulary")
print("=" * 60)
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
print(f"vocab[104] = {vocab[104]}  (letter h)")
print(f"vocab[32]  = {vocab[32]}   (space)")
print(f"vocab[257] = {vocab[257]}  (merged token 'in')")
print(f"vocab[262] = {vocab[262]}  (merged token 'Linux')")
print(f"vocab[263] = {vocab[263]}  (merged token 'Linux ')")

# -- Step 6: Decode --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 6: Decoding tokens back to text")
print("=" * 60)
print("Decoding first 20 compressed token ids:")
print(decode(compressed_training_ids[:20]))
print("\nDecoding the full compressed training text:")
print(decode(compressed_training_ids))

# -- Step 7: Encode then decode --------------------------------------------------
print("\n" + "=" * 60)
print("STEP 7: Encode new text")
print("=" * 60)
phrase = "You cannot learn something without getting your hands dirty"
encoded = encode(phrase)
print(f"Original text: {phrase}")
print(f"Encoded token ids: {encoded}")
print(f"Original length in bytes: {len(list(phrase.encode('utf-8')))}")
print(f"Encoded length in tokens: {len(encoded)}")
print(f"Decoded back: {decode(encoded)}")
print("\n" + "=" * 60)
print("STEP 7b: Encoding a word not seen in training")
print("=" * 60)
unseen = "Linux kernel"
encoded_unseen = encode(unseen)
print(f"Text: {unseen}")
print(f"Encoded: {encoded_unseen}")
print(f"Decoded: {decode(encoded_unseen)}")
print(f"\nTokens above 255 (BPE learned): {[t for t in encoded_unseen if t > 255]}")
print(f"Tokens below 256 (raw bytes):    {[t for t in encoded_unseen if t < 256]}")

STEP 1: Text to raw bytes
Text: hello world
Raw bytes: [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]

STEP 2: Finding frequent pairs
Input ids: [1, 2, 3, 1, 2]
Pair counts: {(1, 2): 2, (2, 3): 1, (3, 1): 1}

STEP 3: Merging a pair
Before merge: [1, 2, 3, 1, 2]
After merging (1,2) -> 99: [99, 3, 99]

STEP 4: Training on real text
Training text length in raw bytes: 507
Running 20 merges...

Merge  1: (101, 32) -> 256  (length now 489)
Merge  2: (105, 110) -> 257  (length now 476)
Merge  3: (115, 32) -> 258  (length now 467)
Merge  4: (76, 257) -> 259  (length now 459)
Merge  5: (259, 117) -> 260  (length now 451)
Merge  6: (116, 104) -> 261  (length now 443)
Merge  7: (260, 120) -> 262  (length now 436)
Merge  8: (262, 32) -> 263  (length now 429)
Merge  9: (100, 32) -> 264  (length now 422)
Merge 10: (261, 256) -> 265  (length now 415)
Merge 11: (121, 32) -> 266  (length now 409)
Merge 12: (111, 102) -> 267  (length now 403)
Merge 13: (111, 110) -> 268  (length now 397)
Merge 1